In [ ]:
import sys 
sys.path.append('src')
from sr_model import *

import os
from datasets import load_from_disk, concatenate_datasets
from datasets import Dataset
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader 
import joblib

In [ ]:
dataset = load_from_disk('/home/rsun@ZHANGroup.local/rna_pretrain/hf_data/allen_23_training_new')

## load pretrained model

In [ ]:
import sys
import os
import argparse
import yaml
sys.path.append('src')
from sr_model import single_sr
from datasets import load_from_disk

def load_config(config_path):
    """Load YAML config file"""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

config = load_config('/home/rsun@ZHANGroup.local/solid-recover/configs/rna_3.yaml')
print("Loaded config:")
print(yaml.dump(config, default_flow_style=False))



In [ ]:
feature_num = dataset[0]['feature'].shape[0]
# === Model ===
model_cfg = config['model']
rna_model = single_sr(
    feature_num=feature_num,
    hidden_params=model_cfg['hidden_params'],
    embed_dim=model_cfg['embed_dim'],
    use_rmsnorm=model_cfg['use_rmsnorm'],
    use_residual=model_cfg['use_residual'],
    dropout_p=model_cfg['dropout_p'],
    vae_model=model_cfg['vae_model']
)

### load model weights

In [ ]:
rna_model.init_model(checkpoint_path= '/home/rsun@ZHANGroup.local/solid-recover/runs/rna_vae_residual_1_20250912_1350/models/ckpt_20001.pth')

### add classifiers

In [ ]:
encoder_dir = '../rna_pretrain/anno_data/label_encoder'
labelencoder_dic = {}
for key in ['class',  'neurotransmitter']:
    labelencoder_dic[key] = f'{encoder_dir}/{key}_label_encoder.joblib'
rna_model.add_classifiers(labelencoder_dic)


### train classifiers

In [ ]:
rna_model.train_classifiers(dataset= dataset,
                            project_dir = '/home/rsun@ZHANGroup.local/solid-recover/runs/rna_vae_residual_1_20250912_1350',
                            lr_dic = 1e-3,
                            test_size = 2.5e-3,
                            train_steps = 10000,
                            eval_points = 100,
                            batch_size = 8192,
                            device  = 'cuda',
                            random_seed=42)

### classification

In [ ]:
import scanpy as sc 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## load classifier
#rna_model.load_classifiers('/home/rsun@ZHANGroup.local/solid-recover/runs/rna_vae_residual_1_20250912_1350/classification_models_20250915-084706/classifier_adapter_{steps}.pth')
#rna_model.load_classifiers('/home/rsun@ZHANGroup.local/solid-recover/runs/rna_vae_residual_1_20250912_1350/classification_models_20250915-163953/classifier_adapter_8000.pth')
#rna_model.load_classifiers('/home/rsun@ZHANGroup.local/solid-recover/runs/rna_vae_residual_1_20250912_1350/classification_models_20250916-071813/classifier_adapter_10000.pth')
#rna_model.load_classifiers('/home/rsun@ZHANGroup.local/solid-recover/runs/rna_vae_residual_1_20250912_1350/classification_models_20250915-123656/classifier_adapter_10000.pth')
#rna_model.load_classifiers('/home/rsun@ZHANGroup.local/solid-recover/runs/rna_vae_residual_1_20250912_1350/classification_models_20250915-143004/classifier_adapter_5000.pth')
rna_model.load_classifiers('/home/rsun@ZHANGroup.local/solid-recover/runs/rna_vae_residual_1_20250912_1350/classification_models_20250915-163953/classifier_adapter_8000.pth')
scdata = sc.read_h5ad('../sly_data/qn_analysis/mus_raw.h5ad')
scdata

In [ ]:
sc.pp.normalize_total(scdata , target_sum=1e4)
sc.pp.log1p(scdata)

scdata = rna_model.classify(scdata, 'cuda')

rna_model.set_loss(beta = 1.0)

scdata = rna_model.get_embedding(scdata, ['z_mu'], device = 'cuda')
scdata

In [ ]:
sc.pp.neighbors(scdata, use_rep = 'sr_z_mu')
scdata.obsm['scvi_umap'] = scdata.obsm['X_umap'].copy()
sc.tl.umap(scdata, min_dist = 0.4)#, key_added = 'sr_umap' )
scdata.obsm['sr_umap'] = scdata.obsm['X_umap'].copy()
sc.pl.embedding(scdata, basis = 'sr_umap', color = [ 'pred_class','pred_neurotransmitter'], ncols=1)

In [ ]:
sc.pl.embedding(scdata, basis = 'scvi_umap', color = [ 'pred_class','pred_neurotransmitter'], ncols=1)#,'_scvi_labels'])  ## scvi umap

In [ ]:
rna_model.classify(scdata, pred_probability=True)

In [ ]:
root_dir = 'qn_pred'
save_dir = os.path.join(root_dir, 'vae_8000')
os.makedirs(save_dir, exist_ok=True)
scdata.obs.to_csv(os.path.join(save_dir, 'mus_obs.csv'))
scdata.uns['pred_prob_class'].to_csv(os.path.join(save_dir, 'prob_class.csv'))
scdata.uns['pred_prob_neurotransmitter'].to_csv(os.path.join(save_dir, 'prob_neuron.csv'))